# What is an experiment worth before you run it?

Two designs are on the table. A cluster holdout: dark a fifth of the map for eight weeks,
and it costs you the revenue of a fifth of the map for eight weeks. A ghost bid: serve the
auction and log the counterfactual, and it costs almost nothing. Both answer the same
question — *how much does this treatment move the outcome?* — and everybody in the room
already knows the holdout is the better experiment.

Better by how much, though? That is the question that decides whether the holdout is worth
five times the price, and "it's more rigorous" is not an answer to it.

**Without this module**, that argument gets settled by seniority, and the loser is usually
the budget: teams buy the big experiment, wait a quarter, and find the posterior barely
moved — because the prior was already tight, or the design's standard error was never going
to touch it. `axiom.design` prices an experiment in two currencies before it is run: **nats**
of expected information, and the **numeraire** of the decision that is waiting on it.

In [ ]:
import numpy as np

from axiom.core import Unsupported
from axiom.design import (
    DESIGN_RELATIVE_SE, DecisionSpec, EIGEstimate, EVOIResult, ReExperimentTiming, ValueFn,
    decayed_sd, eig_gaussian, eig_monte_carlo, evoi_gaussian, evpi, evpi_gaussian, evsi,
    evsi_gaussian, experiment_se_for_design, information_half_life, preposterior_sd,
    preposterior_sd_ratio, time_to_re_experiment,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import ORANGE, MUTED, annotate, caption, compare, curve_band, lines, mark_x, shade

enable();  # every axiom result renders itself from here on

## A design kind has a price in standard error

`DESIGN_RELATIVE_SE` is the table of what each family of design typically achieves, as a
standard error *relative to the value being measured*. Multiply by the reference value and
you have the experiment's standard error without having run it.

A kind that is not in the table returns `Unsupported` rather than a plausible default —
guessing here would put a made-up number at the head of a budget argument.

In [ ]:
print("relative SE by design kind:", dict(DESIGN_RELATIVE_SE))
se_holdout = experiment_se_for_design("cluster_holdout", reference_value=2.0)
se_ghost = experiment_se_for_design("ghost", reference_value=2.0)
print("cluster holdout se:", se_holdout, "| ghost se:", se_ghost)
print("unknown kind ->", type(experiment_se_for_design("crossover", 2.0)).__name__)
print("with explicit relative_se:", experiment_se_for_design("crossover", 2.0, relative_se=0.15))

In [ ]:
prior_sd = 0.5
table(
    [[f"{se:.2f}", f"{eig_gaussian(prior_sd, se):.4f}"] for se in (se_holdout, se_ghost, 1.0)],
    headers=("experiment se", "EIG (nats)"),
)

## The exchange rate is a logarithm, and that is the whole point

    EIG = 0.5 · ln(1 + prior_sd² / experiment_se²)   nats

Information is bought at a *diminishing* rate. The first halving of an experiment's standard
error is worth a lot; the fourth is worth almost nothing, because by then the prior — not
the experiment — is what limits you. The curve below is the reason a design conversation
should start with a number rather than with adjectives.

In [ ]:
se_grid = np.linspace(0.05, 1.2, 120)
fig = curve_band(
    se_grid, [eig_gaussian(prior_sd, se) for se in se_grid],
    label="EIG",
    title="What a nat costs",
    subtitle=f"one parameter, prior sd {prior_sd} — expected information against the design's standard error",
    x_title="experiment standard error", y_title="expected information gain (nats)",
)
mark_x(fig, se_holdout, text=f"cluster holdout · {eig_gaussian(prior_sd, se_holdout):.2f} nats")
mark_x(fig, se_ghost, text=f"ghost · {eig_gaussian(prior_sd, se_ghost):.2f} nats", color=ORANGE)
shade(fig, 0.05, prior_sd / 3, text="prior-limited", color=MUTED)
caption(fig, "The holdout is the better experiment and always will be. The question is whether "
             "the gap in nats is worth the gap in price — and now the gap in nats is a number.")

## Your prior is not Gaussian, so there is a Monte-Carlo route

Real priors come out of `axiom.meta` as draws, not as a standard deviation: skewed, sometimes
bimodal, occasionally truncated at zero because a negative effect is physically impossible.
`eig_monte_carlo` is the nested estimator for a prior given as draws, and it carries its own
standard error so you can tell a converged answer from a noisy one.

The check that matters is that it agrees with the closed form on the one case where the
closed form is exact. The bias is `O(1 / n_prior_draws)`.

In [ ]:
rng = np.random.default_rng(0)
prior_draws = rng.normal(0.0, prior_sd, size=4000)
closed = eig_gaussian(prior_sd, 0.3)
rows = []
for n_sims in (200, 1000, 4000):
    est: EIGEstimate = eig_monte_carlo(prior_draws, 0.3, n_sims=n_sims, seed=1)
    rows.append([n_sims, f"{est.eig:.4f}", f"{est.se:.4f}", f"{closed:.4f}"])
table(rows, headers=("n_sims", "EIG", "MC se", "closed form"))

In [ ]:
sims = (200, 500, 1000, 2000, 4000)
ests = [eig_monte_carlo(prior_draws, 0.3, n_sims=n, seed=1) for n in sims]
fig = curve_band(
    sims, [e.eig for e in ests],
    [e.eig - 2 * e.se for e in ests], [e.eig + 2 * e.se for e in ests],
    label="Monte Carlo",
    title="The estimator earns its error bar",
    subtitle="nested MC against the closed form it must reproduce, ±2 MC standard errors",
    x_title="simulated experiments", y_title="EIG (nats)",
)
curve_band(sims, [closed] * len(sims), label="closed form", color=ORANGE, dash="dot", fig=fig)
caption(fig, "The band is the estimator's own honesty: it narrows with n_sims and it "
             "contains the exact answer at every n. An EIG reported without one is a guess.")

## Information ages, and a stale experiment is a decision made on nothing

The world moves. The posterior an experiment bought degrades — in this model the posterior
*variance* doubles every half-life, so `decayed_sd(sd, t, hl) = sd · 2^(t / 2hl)`. That has a
practical consequence most measurement programmes learn the expensive way: an experiment run
once is a depreciating asset, and `time_to_re_experiment` is the date it needs replacing.

In [ ]:
post_sd = 0.12
print("decayed sd after 0, 6, 12 periods (hl=6):", [round(decayed_sd(post_sd, t, 6.0), 4) for t in (0, 6, 12)])
print("information half-life:", round(information_half_life(prior_sd, post_sd, 6.0), 2), "periods")
timing: ReExperimentTiming = time_to_re_experiment(post_sd, 6.0, experiment_se=0.2, min_eig=0.3, design_kind="cluster_holdout")
print(f"EIG now {timing.eig_now:.3f}; worth {timing.min_eig} nats again after {timing.periods:.1f} periods (sd {timing.sd_at_threshold:.3f})")

In [ ]:
periods = np.linspace(0.0, 24.0, 120)
fig = lines(
    periods,
    {
        "knowledge (posterior sd)": [decayed_sd(post_sd, t, 6.0) for t in periods],
        "re-running is worth (nats)": [eig_gaussian(decayed_sd(post_sd, t, 6.0), 0.2) for t in periods],
    },
    title="An experiment is a depreciating asset",
    subtitle=f"half-life 6 periods — what you know decays, so re-measuring becomes worth paying for again",
    x_title="periods since the experiment", y_title="posterior sd  ·  EIG of a repeat (nats)",
)
mark_x(fig, timing.periods, text=f"re-run at {timing.periods:.1f} periods")
annotate(fig, timing.periods, timing.min_eig, f"back above {timing.min_eig} nats")
caption(fig, "Two curves that cross the same threshold from opposite sides: this is why "
             "`time_to_re_experiment` returns a date and not an opinion.")

## Nats do not appear in a budget. Money does.

Information is only worth what the decision waiting on it is worth. `DecisionSpec` states
that decision plainly: *act* pays `value_per_outcome_unit · (θ − threshold)`, *hold* pays
nothing, so the decision-maker acts when θ is expected to clear the threshold.

- **EVPI** is what a clairvoyant would be worth — the ceiling on any experiment.
- **EVSI** is what *this* experiment is worth, which is less, because it leaves a
  preposterior sd of `prior_sd · sqrt(prior_sd² / (prior_sd² + se²))` behind.

The gap between them is the part of the question your design will not answer, in the
currency the decision is denominated in. Compare EVSI to the price of the design and the
argument that opened this notebook is over.

In [ ]:
decision = DecisionSpec(name="scale_up", threshold=0.0, value_per_outcome_unit=1000.0, numeraire="USD")
mean, sd, se = 0.1, 0.5, 0.3
print("preposterior ratio:", round(preposterior_sd_ratio(sd, se), 4), "| preposterior sd:", round(preposterior_sd(sd, se), 4))
print("EVPI:", round(evpi_gaussian(decision, mean, sd), 2), decision.numeraire)
print("EVSI:", round(evsi_gaussian(decision, mean, sd, se), 2), decision.numeraire)
res: EVOIResult = evoi_gaussian(decision, mean, sd, se)
print(res.model_dump(include={"evpi", "evsi", "preposterior_sd", "method", "numeraire"}))

In [ ]:
ceiling = evpi_gaussian(decision, mean, sd)
fig = curve_band(
    se_grid, [evsi_gaussian(decision, mean, sd, se) for se in se_grid],
    label="EVSI",
    title="What the experiment is worth, in dollars",
    subtitle=f"decision '{decision.name}' at ${decision.value_per_outcome_unit:,.0f} per outcome unit",
    x_title="experiment standard error", y_title=f"value of the experiment ({decision.numeraire})",
)
curve_band(se_grid, [ceiling] * len(se_grid), label="EVPI (clairvoyance)", color=ORANGE, dash="dot", fig=fig)
mark_x(fig, se_holdout, text=f"holdout · ${evsi_gaussian(decision, mean, sd, se_holdout):,.0f}")
mark_x(fig, se_ghost, text=f"ghost · ${evsi_gaussian(decision, mean, sd, se_ghost):,.0f}", color=ORANGE)
caption(fig, "Everything above the EVSI curve is money the experiment cannot earn back, "
             "no matter how well it is run. That number is knowable before the kickoff meeting.")

In [ ]:
draws = rng.normal(mean, sd, size=20000)
print("MC EVPI:", round(evpi(decision, draws), 2), "vs closed", round(evpi_gaussian(decision, mean, sd), 2))
rows = []
for n_sims in (500, 4000):
    mc = evsi(decision, draws, se, n_sims=n_sims, seed=2)
    rows.append([n_sims, f"{mc.evsi:.2f}", f"{mc.evsi_se:.2f}", f"{res.evsi:.2f}"])
table(rows, headers=("n_sims", "MC EVSI", "MC se", "closed form"))

### A payoff that is capped is a different decision

The closed forms assume the payoff is linear in θ forever. Real ones are not: capacity runs
out, a contract caps the upside, a rollout can only reach so many units. `value_fn` takes the
payoff you actually face, and the value of information falls with it — because information
about upside you cannot capture is not worth buying.

In [ ]:
# A non-linear payoff of acting: capped upside.
capped: ValueFn = lambda theta: np.minimum(decision.payoff(theta), 200.0)
print("EVPI with capped payoff:", round(evpi(decision, draws, value_fn=capped), 2))
print("EVSI with capped payoff:", round(evsi(decision, draws, se, n_sims=2000, seed=3, value_fn=capped).evsi, 2))

In [ ]:
caps = (100.0, 200.0, 400.0, 800.0, 1600.0)
fig = lines(
    [np.log10(c) for c in caps],
    {
        "EVPI (clairvoyance)": [evpi(decision, draws, value_fn=lambda t, c=c: np.minimum(decision.payoff(t), c)) for c in caps],
        "EVSI (this design)": [evsi(decision, draws, se, n_sims=2000, seed=3,
                                    value_fn=lambda t, c=c: np.minimum(decision.payoff(t), c)).evsi for c in caps],
    },
    title="A cap on the upside is a cap on what knowing is worth",
    subtitle="same prior, same experiment — only the payoff of acting changes",
    x_title="log₁₀ cap on the payoff of acting (USD)", y_title=f"value of information ({decision.numeraire})",
)
caption(fig, "Both curves flatten where the cap stops binding. Below that, an experiment "
             "priced against the uncapped decision is being over-bought.")

## What this bought you

Before anyone books a holdout window, you can state: this design buys **0.4 nats**, that one
buys **0.1**, the decision is worth **$X** with certainty and **$Y** with this experiment,
and the whole thing needs re-running in **eleven periods**. Every one of those is a number
with an interval and a stated model behind it, computed from a prior and a standard error —
no sampler, no fitted model, no data yet.

Next: `03-methods.ipynb` turns a design *kind* into a concrete allocation, and
`06-economics-and-portfolio.ipynb` runs this comparison across a whole slate of experiments
competing for the same budget.